# QFF2026 Phase 2 — PS-Q2 and common tasks

This notebook uses the official Q2 training and unlabeled test-feature files. Q2 has no validation partition, so hyperparameters are selected with stratified cross-validation on a stratified training subset only. Test labels are unavailable and are never used. The four-qubit feature map is evaluated under ideal simulation and the unmodified QFF2026-v2 noise model.

Run from the repository root in the dedicated Python 3.11–3.13 environment.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "processor_d_config.json").is_file(), "Run this notebook from the repository root."
(ROOT / "results" / "phase2").mkdir(parents=True, exist_ok=True)

## Task A — Bell-state fidelity

Use the official 8,192-shot `task_a_bell()` implementation. Fidelity is computed from XX, YY and ZZ correlations as specified in the kit.

In [2]:
from benchmark.qff_qml_benchmark import task_a_bell

task_a_results = task_a_bell()
(ROOT / "results" / "phase2" / "task_a_results.json").write_text(
    json.dumps(task_a_results, indent=2) + "\n", encoding="utf-8"
)
task_a_results

{'shots': 8192,
 'noise_model_version': 'QFF2026-v2',
 'ideal': {'XX': 1.0,
  'XX_se': 0.0,
  'YY': -1.0,
  'YY_se': 0.0,
  'ZZ': 1.0,
  'ZZ_se': 0.0,
  'fidelity': 1.0,
  'fidelity_se': 0.0},
 'noisy': {'XX': 0.819091796875,
  'XX_se': 0.006338134717392098,
  'YY': -0.827880859375,
  'YY_se': 0.0061971720755083825,
  'ZZ': 0.87353515625,
  'ZZ_se': 0.005378000738451436,
  'fidelity': 0.880126953125,
  'fidelity_se': 0.0025920525815950387}}

## Task B — official Processor D QFT-3 benchmark

This runs the kit's official command and saves its unmodified output for the report. The output must match `benchmark_results` in `processor_d_config.json`.

In [3]:
benchmark_run = subprocess.run(
    [sys.executable, "benchmark/qff_qml_benchmark.py", "processor_d_config.json"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
benchmark_output = benchmark_run.stdout
print(benchmark_output)
(ROOT / "results" / "phase2" / "processor_d_benchmark.txt").write_text(
    benchmark_output, encoding="utf-8"
)

Processor D design is valid. Official QFT-3 benchmark_results:
{
  "qft3_circuit_depth_before_transpile": 6,
  "qft3_circuit_depth_after_transpile": 18,
  "qft3_two_qubit_gate_count": 9,
  "qft3_swap_overhead": 0,
  "qft3_fidelity_ideal": 1.0,
  "qft3_fidelity_noisy": 0.668377,
  "benchmark_noise_model_version": "QFF2026-v2"
}



329

## PS-Q2 — quantum kernel and classical RBF-SVM

The training subset and all preprocessing/model selection are confined to `hd_data_train.csv`. Both models use the same stratified subset and folds of the first six features identified by the official feature dictionary. Each fold fits its transformations on its training rows only: the four-qubit quantum model uses four PCA components, while the RBF-SVM baseline retains all six standardized signal features. The official Q2 package supplies no validation data. No test labels are loaded or inferred.

In [4]:
from q2_solution import run_experiment

q2_results = run_experiment()
q2_results

{'dataset': {'training_rows_available': 800,
  'training_subset_rows': 200,
  'training_subset_class_counts': {'A': 97, 'B': 103},
  'test_rows': 200,
  'test_ids_preserved': True,
  'test_labels_used': False,
  'features_used_for_reduction': ['x01', 'x02', 'x03', 'x04', 'x05', 'x06'],
  'pca_components': 4,
  'classical_baseline_features': ['x01', 'x02', 'x03', 'x04', 'x05', 'x06']},
 'model_selection': {'method': '3-fold stratified cross-validation on training subset only',
  'quantum_selected_C': 100.0,
  'classical_selected_parameters': {'kernel': 'RBF', 'C': 1.0, 'gamma': 0.3},
  'balanced_accuracy_cv': {'quantum_ideal': 0.5148146379764027,
   'quantum_noisy': 0.509912677192089,
   'classical_rbf': 0.607845789831084},
  'quantum_cv_scores_by_C': {'ideal': {'0.01': 0.5,
    '0.1': 0.5,
    '1.0': 0.4736562049062049,
    '10.0': 0.5014827900857313,
    '100.0': 0.5148146379764027},
   'noisy': {'0.01': 0.5,
    '0.1': 0.5,
    '1.0': 0.48330309396485865,
    '10.0': 0.49655456879721

The noisy quantum-kernel model produces `predictions_q2.csv` with the original official `sample_id` values and labels `A`/`B`. Balanced-accuracy figures in `results.md` are training-only cross-validation estimates, not test-set scores. The noisy kernel is a normalized Hilbert–Schmidt overlap of density matrices produced under QFF2026-v2; the ideal kernel is the squared pure-state overlap.